In [1]:
import torch
from icecream import ic
import itertools

In [2]:
device = torch.device("cuda")

In [3]:
torch.cuda.get_device_name(0)

'NVIDIA A100 80GB PCIe'

In [4]:
num_hist_dims = 2
extent = torch.tensor([[0.0, 3.0], [0.0, 4.0]]).to(device)
histogram_shape = (500, 500)
positions = torch.rand(100, 100_000, 2).to(device)
charges = torch.ones_like(positions[..., 0])
bin_widths = (extent[..., 1] - extent[..., 0]) / (torch.tensor(
    histogram_shape, device=positions.device
) - 1)
inside_mask = ((extent[..., 0] <= positions) & (positions < extent[..., 1])).all(
    dim=-1
)  # Shape (..., num_samples)
masked_charges = charges * inside_mask  # Shape (..., num_samples)
positions_in_bin_space = (positions - extent[..., 0]) / bin_widths
positions_in_bin_space_int_component = positions_in_bin_space.floor().long()
positions_in_bin_space_fractional_components = (
    positions_in_bin_space - positions_in_bin_space_int_component
)

In [5]:
# _ = ic(
#     torch.mps.current_allocated_memory() / 1e6,
#     torch.mps.driver_allocated_memory() / 1e6,
# )

In [6]:
corner_offsets = torch.tensor(list(itertools.product([0, 1], repeat=num_hist_dims)), device=device)

In [7]:
%%timeit
(corner_offsets - positions_in_bin_space_fractional_components.unsqueeze(-2)).abs()

896 μs ± 57.6 ns per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [8]:
%%timeit
corner_weight_factors = torch.where(
    corner_offsets == 0, 
    positions_in_bin_space_fractional_components.unsqueeze(-2),
    1.0 - positions_in_bin_space_fractional_components.unsqueeze(-2),
)  # Shape (..., num_samples, num_corners, num_hist_dims)

53.4 μs ± 25.7 μs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [9]:
%%timeit
# Generate all corner combinations and their weights
corner_offsets = torch.tensor(list(itertools.product([0, 1], repeat=num_hist_dims)), device=device)
corner_positions_in_bin_space = (
    positions_in_bin_space_int_component.unsqueeze(-2) + corner_offsets
).clamp(
    corner_offsets.new_zeros(()),
    corner_offsets.new_tensor(histogram_shape) - 1,
)
corner_weight_factors = (corner_offsets - positions_in_bin_space_fractional_components.unsqueeze(-2)).abs()  # Shape (..., num_samples, num_corners, num_hist_dims)
corner_weights = corner_weight_factors.prod(
    dim=-1
)  # Shape (..., num_samples, num_corners)
corner_charges = (  # Actual charge deposition on the corners
    masked_charges.unsqueeze(-1) * corner_weights
)  # Shape (..., num_samples, num_corners)

getattr(torch, device.type).synchronize()

3.23 ms ± 1.33 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [20]:
%%timeit
# Generate all corner combinations and their weights
corner_offsets = torch.tensor(list(itertools.product([0, 1], repeat=num_hist_dims)), device=device)
corner_positions_in_bin_space = (
    positions_in_bin_space_int_component.unsqueeze(-2) + corner_offsets
).clamp(
    corner_offsets.new_zeros(()),
    corner_offsets.new_tensor(histogram_shape) - 1,
)
positions_in_bin_space_fractional_components.unsqueeze(-2)
corner_weight_factors = torch.where(
    corner_offsets == 0, 
    positions_in_bin_space_fractional_components.unsqueeze(-2),
    (1.0 - positions_in_bin_space_fractional_components).unsqueeze(-2),
)  # Shape (..., num_samples, num_corners, num_hist_dims)
corner_weights = corner_weight_factors.prod(
    dim=-1
)  # Shape (..., num_samples, num_corners)
corner_charges = (  # Actual charge deposition on the corners
    masked_charges.unsqueeze(-1) * corner_weights
)  # Shape (..., num_samples, num_corners)

2.97 ms ± 468 ns per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [21]:
%%timeit
# Generate all corner combinations and their weights
corner_offsets = torch.tensor(list(itertools.product([0, 1], repeat=num_hist_dims)), device=device)
corner_positions_in_bin_space = (
    positions_in_bin_space_int_component.unsqueeze(-2) + corner_offsets
).clamp(
    corner_offsets.new_zeros(()),
    corner_offsets.new_tensor(histogram_shape) - 1,
)
positions_in_bin_space_fractional_components.unsqueeze(-2)
corner_weight_factors = torch.where(
    corner_offsets == 0, 
    positions_in_bin_space_fractional_components.unsqueeze(-2),
    (1.0 - positions_in_bin_space_fractional_components).unsqueeze(-2),
)  # Shape (..., num_samples, num_corners, num_hist_dims)
corner_weights = corner_weight_factors.prod(
    dim=-1
)  # Shape (..., num_samples, num_corners)
corner_charges = (  # Actual charge deposition on the corners
    masked_charges.unsqueeze(-1) * corner_weights
)  # Shape (..., num_samples, num_corners)

2.97 ms ± 327 ns per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [22]:
%%timeit
# Generate all corner combinations and their weights
corner_offsets = torch.tensor(list(itertools.product([0, 1], repeat=num_hist_dims)), device=device)
corner_positions_in_bin_space = (
    positions_in_bin_space_int_component.unsqueeze(-2) + corner_offsets
).clamp(
    corner_offsets.new_zeros(()),
    corner_offsets.new_tensor(histogram_shape) - 1,
)
corner_weight_factors = torch.where(
    corner_offsets == 0, 
    positions_in_bin_space_fractional_components.unsqueeze(-2),
    1.0 - positions_in_bin_space_fractional_components.unsqueeze(-2),
)  # Shape (..., num_samples, num_corners, num_hist_dims)
corner_weights = corner_weight_factors.prod(
    dim=-1
)  # Shape (..., num_samples, num_corners)
corner_charges = (  # Actual charge deposition on the corners
    masked_charges.unsqueeze(-1) * corner_weights
)  # Shape (..., num_samples, num_corners)

getattr(torch, device.type).synchronize()

3 ms ± 890 ns per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [14]:
%%timeit
# Generate all corner combinations and their weights
num_corners = 2**num_hist_dims
# TODO speed: torch.tensor(list(itertools.product([0, 1], repeat=D)), device=device)
corner_offsets = (
    torch.arange(num_corners).unsqueeze(-1).to(positions.device)
    // (2 ** torch.arange(num_hist_dims).to(positions.device))
    % 2
)
corner_positions_in_bin_space = (
    positions_in_bin_space_int_component.unsqueeze(-2) + corner_offsets
)
clamped_corner_positions_in_bin_space = corner_positions_in_bin_space.clamp(
    corner_positions_in_bin_space.new_zeros(()),
    corner_positions_in_bin_space.new_tensor(histogram_shape) - 1,
)
corner_weight_factors = positions_in_bin_space_fractional_components.unsqueeze(
    -2
).where(
    corner_offsets == 1,
    1.0 - positions_in_bin_space_fractional_components.unsqueeze(-2),
)  # Shape (..., num_samples, num_corners, num_hist_dims)
corner_weights = corner_weight_factors.prod(
    dim=-1
)  # Shape (..., num_samples, num_corners)
corner_charges = (  # Actual charge deposition on the corners
    masked_charges.unsqueeze(-1) * corner_weights
)  # Shape (..., num_samples, num_corners)

getattr(torch, device.type).synchronize()

3.07 ms ± 2.83 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [15]:
# _ = ic(
#     torch.mps.current_allocated_memory() / 1e6,
#     torch.mps.driver_allocated_memory() / 1e6,
# )

In [10]:
del (
    num_hist_dims,
    extent,
    histogram_shape,
    positions,
    charges,
    bin_widths,
    inside_mask,
    masked_charges,
    positions_in_bin_space,
    positions_in_bin_space_int_component,
    positions_in_bin_space_fractional_components,
)

In [11]:
# _ = ic(
#     torch.mps.current_allocated_memory() / 1e6,
#     torch.mps.driver_allocated_memory() / 1e6,
# )

In [12]:
num_dims = 2
positions = (torch.rand(100, 100_000).to(device), torch.rand(100, 100_000).to(device))
weights = torch.ones_like(positions[0]).to(device)
bins = (
    torch.linspace(0.0, 1.0, 500).to(device),
    torch.linspace(0.0, 1.0, 500).to(device),
)
grid_sizes = []
spacings = []
for i, bin_array in enumerate(bins):
    N = bin_array.numel()
    if N < 2:
        raise ValueError(f"bins[{i}] must have at least 2 elements")

    spacing = bin_array[1] - bin_array[0]
    if N > 2:
        diffs = torch.diff(bin_array)
        if not torch.allclose(diffs, spacing, rtol=1e-4):
            raise ValueError(f"bins[{i}] must have uniform spacing")

    grid_sizes.append(N)
    spacings.append(spacing)
# Normalize particle coordinates to grid index space
grid_indices = []
fractional_parts = []
for pos, bin_array, spacing in zip(positions, bins, spacings):
    # Normalized coordinate in grid index space
    u = (pos - bin_array[0]) / spacing

    # Left cell index
    i = torch.floor(u).to(torch.int64)
    grid_indices.append(i)

    # Fractional distance to right cell
    w = u - i
    fractional_parts.append(w)

In [13]:
# _ = ic(
#     torch.mps.current_allocated_memory() / 1e6,
#     torch.mps.driver_allocated_memory() / 1e6,
# )

In [14]:
%%timeit
# Generate all corner combinations and their weights
num_corners = 2**num_dims
corner_indices = []
corner_weights = []

for corner in range(num_corners):
    # Determine which corners to use based on binary representation
    corner_offsets = []
    weight_factors = []

    for dim in range(num_dims):
        if (corner >> dim) & 1:  # Use right cell in this dimension
            corner_offsets.append(1)
            weight_factors.append(fractional_parts[dim])
        else:  # Use left cell in this dimension
            corner_offsets.append(0)
            weight_factors.append(1 - fractional_parts[dim])

    # Calculate indices for this corner
    corner_idx_list = []
    for dim in range(num_dims):
        base_idx = grid_indices[dim]
        offset_idx = (base_idx + corner_offsets[dim]).clamp(0, grid_sizes[dim] - 1)
        corner_idx_list.append(offset_idx)

    # Calculate weight for this corner
    corner_weight = weights
    for weight_factor in weight_factors:
        corner_weight = corner_weight * weight_factor

    corner_indices.append(corner_idx_list)
    corner_weights.append(corner_weight)

getattr(torch, device.type).synchronize()

2.48 ms ± 524 ns per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [15]:
# _ = ic(
#     torch.mps.current_allocated_memory() / 1e6,
#     torch.mps.driver_allocated_memory() / 1e6,
# )

In [16]:
del (
    num_dims,
    positions,
    weights,
    bins,
    grid_sizes,
    spacings,
    i,
    bin_array,
    spacing,
    diffs,
    grid_indices,
    fractional_parts,
    pos,
    # bin_array,
    # spacing,
    u,
    # i,
    w,
)

In [17]:
# _ = ic(
#     torch.mps.current_allocated_memory() / 1e6,
#     torch.mps.driver_allocated_memory() / 1e6,
# )

In [18]:
num_hist_dims = 2
extent = torch.tensor([[0.0, 3.0], [0.0, 4.0]]).to(device)
histogram_shape = (3, 4)
positions = torch.rand(100, 100_000, 2).to(device)
charges = torch.ones_like(positions[..., 0])
bin_widths = (extent[..., 1] - extent[..., 0]) / torch.tensor(
    histogram_shape, device=positions.device
)
inside_mask = ((extent[..., 0] <= positions) & (positions < extent[..., 1])).all(
    dim=-1
)  # Shape (..., num_samples)
masked_charges = charges * inside_mask  # Shape (..., num_samples)
positions_in_bin_space = (positions - extent[..., 0]) / bin_widths
positions_in_bin_space_int_component = positions_in_bin_space.floor().long()
positions_in_bin_space_fractional_components = (
    positions_in_bin_space - positions_in_bin_space_int_component
)

In [19]:
# _ = ic(
#     torch.mps.current_allocated_memory() / 1e6,
#     torch.mps.driver_allocated_memory() / 1e6,
# )

In [20]:
%%timeit
# Generate all corner combinations and their weights
num_corners = 2**num_hist_dims
# TODO speed: torch.tensor(list(itertools.product([0, 1], repeat=D)), device=device)
corner_offsets = (
    torch.arange(num_corners).unsqueeze(-1).to(positions.device)
    // (2 ** torch.arange(num_hist_dims).to(positions.device))
    % 2
)
corner_positions_in_bin_space = (
    positions_in_bin_space_int_component.unsqueeze(-2) + corner_offsets
)
clamped_corner_positions_in_bin_space = corner_positions_in_bin_space.clamp(
    corner_positions_in_bin_space.new_zeros(()),
    corner_positions_in_bin_space.new_tensor(histogram_shape) - 1,
)
corner_weight_factors = positions_in_bin_space_fractional_components.unsqueeze(
    -2
).where(
    corner_offsets == 1,
    1.0 - positions_in_bin_space_fractional_components.unsqueeze(-2),
)  # Shape (..., num_samples, num_corners, num_hist_dims)
corner_weights = corner_weight_factors.prod(
    dim=-1
)  # Shape (..., num_samples, num_corners)
corner_charges = (  # Actual charge deposition on the corners
    masked_charges.unsqueeze(-1) * corner_weights
)  # Shape (..., num_samples, num_corners)

getattr(torch, device.type).synchronize()

3.07 ms ± 1.68 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [21]:
# _ = ic(
#     torch.mps.current_allocated_memory() / 1e6,
#     torch.mps.driver_allocated_memory() / 1e6,
# )

In [22]:
del (
    num_hist_dims,
    extent,
    histogram_shape,
    positions,
    charges,
    bin_widths,
    inside_mask,
    masked_charges,
    positions_in_bin_space,
    positions_in_bin_space_int_component,
    positions_in_bin_space_fractional_components,
)

In [23]:
# _ = ic(
#     torch.mps.current_allocated_memory() / 1e6,
#     torch.mps.driver_allocated_memory() / 1e6,
# )